### Distillation

In [ ]:
import subprocess
import os
from tqdm.notebook import tqdm

In [ ]:
MODELS = ["dinov2_vitb"]
DATASET = "aqua20"
DATASET_PATH = f"/home/alex/internship/datasets/aqua20"  # Update this path to your dataset location
results = {m: {} for m in MODELS}
DISTILL_CONFIGS = {
    "dinov2_vitb": [{"syn_res": 196, "real_res": 196, "crop_res": 196, "train_crop_mode": "random", "augs_per_batch": 3, "num_eval": 2, "ipc": 1}],
}

In [ ]:
# Aplatir tous les runs à faire
all_runs = [
    (model, cfg)
    for model, configs in DISTILL_CONFIGS.items()
    for cfg in configs
]

for model, config in tqdm(all_runs, desc="Distillation runs", unit="run"):
    run_name = f"{model}_distill_{config['syn_res']}_ipc{config['ipc']}_augs{config['augs_per_batch']}_physics"
    
    tqdm.write(f"\n{'='*50}")
    tqdm.write(f"Distilling {model} | ipc={config['ipc']} | syn_res={config['syn_res']} | augs={config['augs_per_batch']}")
    tqdm.write(f"Run name: {run_name}")
    tqdm.write('='*50)

    env = os.environ.copy()
    env["DATASET"] = DATASET
    env["MODEL"] = model

    process = subprocess.Popen(
        [
            "./run.sh", "distill",
            f"--augs_per_batch={config['augs_per_batch']}",
            f"--syn_res={config['syn_res']}",
            f"--real_res={config['real_res']}",
            f"--crop_res={config['crop_res']}",
            f"--train_crop_mode={config['train_crop_mode']}",
            f"--ipc={config['ipc']}",
            f"--run_name={run_name}",
            f"--data_root={DATASET_PATH}",
            f"--distill_mode=physics_pyramid"
        ],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd="/home/alex/internship/GradientDistillation"
    )

    for line in process.stdout:
        tqdm.write(line, end="")

    process.wait()
    tqdm.write(f"\nReturn code: {process.returncode}")

### Visualization

In [ ]:
import torch
import matplotlib.pyplot as plt

# Load
run_name = "distill_aqua20_h100_seathru_10ipc"
path = "/Users/alex/Developpement/Internship/GradientDistillation"
data = torch.load(f"{path}/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth", weights_only=False, map_location="cpu")
img_numb = 5


images = data["images"]  # (N, C, H, W)
img = images[img_numb].permute(1, 2, 0).cpu().numpy()

# Figure sans bords
fig, ax = plt.subplots(figsize=(img.shape[1] / 100, img.shape[0] / 100), dpi=100)
ax.imshow(img)
ax.axis("off")
ax.set_position([0, 0, 1, 1])  # remplit toute la figure
plt.savefig("outputSeaThru.png", bbox_inches="tight", pad_inches=0, dpi=300)

plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

# --- Load ---
run_name = "distill_aqua20_h100_seathru_10ipc"
path = "/Users/alex/Developpement/Internship/GradientDistillation"
data = torch.load(
    f"{path}/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth",
    weights_only=False, map_location="cpu",
)

images = data["images"]  # (N, C, H, W)
N = images.shape[0]
print(f"Keys: {data.keys()} | Shape: {images.shape}")

# --- AQUA20 class names (ImageFolder alphabetical order) ---
class_names = [
    "coral", "crab", "diver", "eel", "fish",
    "fishInGroups", "flatworm", "jellyfish", "marine_dolphin", "octopus",
    "rayfish", "seaAnemone", "seaCucumber", "seaSlug", "seaUrchin",
    "shark", "shrimp", "squid", "starfish", "turtle",
]
num_classes = len(class_names)

# --- Infer IPC + group image indices by class ---
assert N % num_classes == 0, f"N={N} not divisible by {num_classes} classes"
ipc = N // num_classes
print(f"Detected IPC = {ipc}")

labels = data.get("labels", None)
if labels is not None:
    labels = torch.as_tensor(labels).view(-1).tolist()
    groups = [[i for i in range(N) if labels[i] == c] for c in range(num_classes)]
else:
    # class-major ordering: image idx -> class idx // ipc
    groups = [[c * ipc + j for j in range(ipc)] for c in range(num_classes)]

# --- Per-image normalization to [0, 1] for display ---
def to_displayable(img):
    img = img.float()
    lo, hi = img.amin(), img.amax()
    img = (img - lo) / (hi - lo + 1e-8)
    return img.permute(1, 2, 0).cpu().numpy()

# --- Grid: rows = classes, cols = IPC variants ---
nrows, ncols = num_classes, ipc
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 1.6, nrows * 1.6),
    squeeze=False,
)

for c in range(num_classes):
    for j in range(ncols):
        ax = axes[c][j]
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        if j < len(groups[c]):
            ax.imshow(to_displayable(images[groups[c][j]]))
        if j == 0:
            ax.set_ylabel(
                class_names[c], rotation=0, ha="right", va="center", fontsize=10
            )

fig.subplots_adjust(left=0.12, right=0.98, top=0.99, bottom=0.01,
                    wspace=0.05, hspace=0.10)

plt.savefig(f"{path}/distilled_grid.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt

run_name = "dinov2_vitb_distill_196_ipc1_augs3_physics"
data = torch.load(
    f"/home/alex/internship/GradientDistillation/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth",
    weights_only=False,
)
print(f"Keys in data: {data.keys()}")

img_to_show = 1

J_1 = data["syn_J"][img_to_show]    # (3, H, W)  clean radiance
T_1 = data["syn_T"][img_to_show]    # (1, H, W)  transmission
B_1 = data["syn_B"][img_to_show]    # (3, 1, 1)  background light
I_1 = data["images"][img_to_show]   # (3, H, W)  observed

print("shapes:", J_1.shape, T_1.shape, B_1.shape, I_1.shape)
print("ranges:")
for name, t in [("I", I_1), ("J", J_1), ("T", T_1), ("B", B_1)]:
    print(f"  {name}: min={t.min().item():.3f}  max={t.max().item():.3f}  mean={t.mean().item():.3f}")


def img_to_np(t, normalize=True):
    """(C, H, W) -> (H, W, C) numpy for imshow. Preserves colour by joint min-max."""
    t = t.detach().cpu().float()
    if normalize:
        t = (t - t.min()) / (t.max() - t.min() + 1e-8)
    if t.shape[0] == 1:                # grayscale
        return t.squeeze(0).numpy()
    return t.permute(1, 2, 0).numpy()  # (H, W, 3)


def b_to_swatch(B, size=128, normalize=True):
    """(3, 1, 1) or (3,) -> (size, size, 3) numpy swatch."""
    B = B.detach().cpu().float().squeeze()   # -> (3,)
    swatch = B[:, None, None].expand(3, size, size).contiguous()
    if normalize:
        swatch = (swatch - swatch.min()) / (swatch.max() - swatch.min() + 1e-8)
    return swatch.permute(1, 2, 0).numpy()


fig, axes = plt.subplots(2, 2, figsize=(10, 10))

axes[0, 0].imshow(img_to_np(I_1))
axes[0, 0].set_title(f"I — observed  {tuple(I_1.shape)}")
axes[0, 0].axis("off")

axes[0, 1].imshow(img_to_np(J_1))
axes[0, 1].set_title(f"J — clean radiance  {tuple(J_1.shape)}")
axes[0, 1].axis("off")

axes[1, 0].imshow(img_to_np(T_1), cmap="gray")
axes[1, 0].set_title(f"T — transmission  {tuple(T_1.shape)}")
axes[1, 0].axis("off")

b_vals = torch.sigmoid(B_1).detach().cpu().float().squeeze().tolist()
axes[1, 1].imshow(b_to_swatch(B_1))
axes[1, 1].set_title(f"B — background  rgb=[{b_vals[0]:.2f}, {b_vals[1]:.2f}, {b_vals[2]:.2f}]")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils

# Load
run_name = "dinov2_vitb_distill_196_ipc1_augs3_physics"
data = torch.load(f"/home/alex/internship/GradientDistillation/logged_files/distillation/aqua20/dinov2_vitb/{run_name}/data.pth", weights_only=False)

images = data["images"]  # (N, C, H, W)
print(f"Shape: {images.shape}")

# Display grid
grid = vutils.make_grid(images, nrow=4, padding=2)
plt.figure(figsize=(12, 12))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.title("Distilled Images")
plt.show()

### Benchmark

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
import time
from tqdm.notebook import tqdm
from torch.utils.data import Subset
from collections import defaultdict
import random

In [ ]:
DATA_ROOT = "/home/alex/internship/datasets/aqua20/data/aqua20"
DISTILLED_PTH = "../logged_files/distillation/aqua20/dinov2_vitb/dinov2_vitb_distill_196_ipc1_augs3_physics/data.pth"
NUM_CLASSES = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESOLUTION = 196

In [ ]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
backbone = backbone.to(DEVICE)

In [ ]:
def extract_features(loader, desc="Extracting features"):
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in tqdm(loader, desc=desc, leave=False):
            all_feats.append(backbone(x.to(DEVICE)).cpu())
            all_labels.append(y)
    return torch.cat(all_feats), torch.cat(all_labels)


def train_linear_probe(train_feats, train_labels, test_feats, test_labels,
                       epochs=50, lr=1e-3, eval_every=10):
    head = nn.Linear(768, NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Loaders sur features précalculées — tout en RAM, très rapide
    train_feat_loader = DataLoader(
        TensorDataset(train_feats, train_labels), batch_size=256, shuffle=True
    )
    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    for epoch in tqdm(range(epochs), desc="Training"):
        head.train()
        total_loss, correct, total = 0.0, 0, 0

        for feats, y in train_feat_loader:
            feats, y = feats.to(DEVICE), y.to(DEVICE)
            logits = head(feats)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(y)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(y)

        if (epoch + 1) % eval_every == 0 or epoch == epochs - 1:
            head.eval()
            all_preds, all_labels_val = [], []
            with torch.no_grad():
                for feats, y in test_feat_loader:
                    feats = feats.to(DEVICE)
                    preds = head(feats).argmax(dim=1).cpu()
                    all_preds.append(preds)
                    all_labels_val.append(y)

            all_preds      = torch.cat(all_preds).numpy()
            all_labels_val = torch.cat(all_labels_val).numpy()

            f1_macro = f1_score(all_labels_val, all_preds, average="macro")
            f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

            tqdm.write(
                f"Epoch {epoch+1:>3}/{epochs} | "
                f"Loss: {total_loss/total:.4f} | "
                f"Train Acc: {correct/total*100:.1f}% | "
                f"F1 Macro: {f1_macro*100:.1f}% | "
                f"F1 Weighted: {f1_weighted*100:.1f}%"
            )

    return head

In [ ]:
full_train = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=transform)
test_ds    = datasets.ImageFolder(f"{DATA_ROOT}/test",  transform=transform)

full_loader = DataLoader(full_train, batch_size=64, shuffle=True, num_workers=4)
test_loader = DataLoader(test_ds,   batch_size=64, shuffle=False, num_workers=4)

In [ ]:
distilled = torch.load(DISTILLED_PTH, map_location=DEVICE)
# distilled est un tensor (N, C, H, W) ou un dict selon ton format
# adapte selon ce que run.sh sauvegarde
print(f"Distilled data keys: {distilled.keys()}")
images_d = distilled["images"].to(DEVICE)   # shape: (20, 3, 196, 196)
labels_d = distilled["labels"].to(DEVICE)
print(labels_d)
from torch.utils.data import TensorDataset
distill_loader = DataLoader(
    TensorDataset(images_d.cpu(), labels_d.cpu()),
    batch_size=20, shuffle=True
)

In [ ]:
ipc = len(images_d) // len(full_train.classes)
rng = random.Random(123)

class_to_indices = defaultdict(list)
for idx, (_, label) in enumerate(full_train.samples):
    class_to_indices[label].append(idx)



stratified_indices = []
for label, indices in class_to_indices.items():
    stratified_indices.extend(rng.sample(indices, ipc))

random_subset = Subset(full_train, stratified_indices)

random_loader = DataLoader(
    random_subset,
    batch_size=len(stratified_indices),
    shuffle=True,
    num_workers=4,
)

In [ ]:
print("Extracting features...")
test_feats,    test_labels    = extract_features(test_loader,    "Test")


#### Full data


In [ ]:
print("\nTraining on full data...")
start_time = time.time()
full_feats,    full_labels    = extract_features(full_loader,    "Full train")
head_full = train_linear_probe(full_feats, full_labels, test_feats, test_labels,
                                epochs=50, eval_every=10)
print(f"Training time: {time.time() - start_time:.6f} seconds")

#### Distilled data


In [ ]:
print("\nTraining on distilled data...")
start_time = time.time()
distill_feats, distill_labels = extract_features(distill_loader, "Distilled")
head_dist = train_linear_probe(distill_feats, distill_labels, test_feats, test_labels,
                                epochs=50, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

#### Random Data


In [ ]:
print("\nTraining on random data...")
start_time = time.time()
random_feats, random_labels = extract_features(random_loader, "Distilled")
head_dist = train_linear_probe(random_feats, random_labels, test_feats, test_labels,
                                epochs=50, eval_every=20)
print(f"Training time: {time.time() - start_time:.6f} seconds")

#### No pretraining (random head)

In [ ]:
def evaluate_random_head(test_feats, test_labels, feat_dim=768):
    head = nn.Linear(feat_dim, NUM_CLASSES).to(DEVICE)
    head.eval()

    test_feat_loader = DataLoader(
        TensorDataset(test_feats, test_labels), batch_size=256, shuffle=False
    )

    all_preds, all_labels_val = [], []
    with torch.no_grad():
        for feats, y in test_feat_loader:
            feats = feats.to(DEVICE)
            preds = head(feats).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_labels_val.append(y)

    all_preds      = torch.cat(all_preds).numpy()
    all_labels_val = torch.cat(all_labels_val).numpy()

    acc         = (all_preds == all_labels_val).mean()
    f1_macro    = f1_score(all_labels_val, all_preds, average="macro")
    f1_weighted = f1_score(all_labels_val, all_preds, average="weighted")

    print(
        f"Random head baseline | "
        f"Test Acc: {acc*100:.1f}% | "
        f"F1 Macro: {f1_macro*100:.1f}% | "
        f"F1 Weighted: {f1_weighted*100:.1f}%"
    )
    return head

print("\nBaseline: random (untrained) head...")
evaluate_random_head(test_feats, test_labels)